In [4]:
import os, pickle
import numpy as np
import pandas as pd
from recsys_utils import fix_text, als_user_recs, save_als_factors

PROC = '../data/processed'
split = pickle.load(open(f'{PROC}/split.pkl', 'rb'))
trainval = split['trainval']
user_ids = split['user_ids']
listings_clean = pd.read_pickle(f'{PROC}/listings_clean.pkl')

FACTORS = '../models/als_factors.npz'
if not os.path.exists(FACTORS):
    print("ALS factors are missing; training the model (~15–20 sec)...")
    from implicit.als import AlternatingLeastSquares
    als_model = AlternatingLeastSquares(factors=128, regularization=0.01, alpha=100.0,
                                        iterations=15, random_state=42)
    als_model.fit(trainval)
    os.makedirs('../models', exist_ok=True)
    save_als_factors(als_model, FACTORS)
    print("ALS Model Trained and Saved.")

f = np.load(FACTORS)
user_factors, item_factors = f['user_factors'], f['item_factors']
print("User:", len(user_ids), "| Houses:", item_factors.shape[0])

User: 140786 | Houses: 279599


In [5]:
SHOW = ['listing_id', 'name', 'city', 'price', 'room_type']

def get_user_recommendations(reviewer_id, N=5):
    row = int(np.searchsorted(user_ids, reviewer_id))
    if row >= len(user_ids) or user_ids[row] != reviewer_id:
        raise KeyError(f"reviewer_id={reviewer_id} Not in the Matrix (requires ≥3 reviews)")

    history_idx = trainval.indices[trainval.indptr[row]:trainval.indptr[row + 1]]
    rec_idx, scores = als_user_recs(row, user_factors, item_factors, trainval, k=N)

    history = listings_clean.iloc[history_idx][SHOW].copy()
    recs = listings_clean.iloc[rec_idx][SHOW].copy()
    recs['score'] = scores
    for d in (history, recs):
        d['name'] = d['name'].map(fix_text)
    return history, recs

In [7]:
REVIEWER_ID = None   
if REVIEWER_ID is None:
    REVIEWER_ID = int(user_ids[split['eval_users'][0]])

try:
    history, recs = get_user_recommendations(REVIEWER_ID, N=5)
    print(f"=== reviewer_id: {REVIEWER_ID} — PREVIOUS HISTORY ===")
    display(history)
    print("\n=== ALS-RECOMMENDED HOUSES (TOP 5) ===")
    display(recs)
except KeyError as e:
    print("❌", e)

=== reviewer_id: 21 — PREVIOUS HISTORY ===


,listing_id,name,city,price,room_type
225506,729898,HEART OF CONDESA ART DECO BUILDING,Mexico City,1322,Entire place
231333,23401489,Super elegant and cozy apartment located in La...,Mexico City,1652,Entire place



=== ALS-RECOMMENDED HOUSES (TOP 5) ===


,listing_id,name,city,price,room_type,score
69052,3916192,Pretty Pretty Pretty Apt Indoor-Outdoor Space ...,Mexico City,2065,Entire place,0.367843
222820,4801896,"Roma/Condesa, Apt 2 habitaciones",Mexico City,1999,Entire place,0.346589
72942,1555819,New Suite in Vintage Home in Roma/Condesa,Mexico City,1100,Entire place,0.324081
222179,246057,HIP APARTMENT IN ZONA ROSA!! SUPER LOCATION!!,Mexico City,867,Entire place,0.307991
261397,997247,NEW! HISTORIC BUILDING -BEST STREET,Mexico City,1283,Entire place,0.305587
